In [ ]:
# ============================================================
# GOOGLE COLAB SETUP — run this cell first when using Colab
# ============================================================
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # 1. Mount Google Drive so your data is accessible
    from google.colab import drive
    drive.mount('/content/drive')

    # 2. Set REPO_PATH to wherever you stored (or will store) the repo on Drive.
    #    If the folder doesn't exist the repo is cloned there automatically.
    REPO_PATH = '/content/drive/MyDrive/Factor-Research'

    if not os.path.exists(REPO_PATH):
        print('Cloning repository to Google Drive...')
        os.system(f'git clone https://github.com/mbrennan5/Factor-Research.git {REPO_PATH}')
    else:
        print(f'Repository found at {REPO_PATH}')

    # 3. Install required packages.
    #    Most are already in Colab; only the non-standard ones need installing.
    print('Installing packages...')
    os.system('pip install -q lightgbm xgboost optuna plotly tqdm yfinance')

    # 4. Change to the notebooks directory so relative paths (../data/...) work.
    NOTEBOOKS_DIR = os.path.join(REPO_PATH, 'notebooks')
    os.chdir(NOTEBOOKS_DIR)
    print(f'Working directory set to: {os.getcwd()}')
else:
    print('Running locally — no Colab setup needed.')


# Alpha Factor Generation 2 - High-Frequency Technical Factors

This notebook generates **advanced alpha factors** from high-frequency intraday trading data, focusing on **market microstructure** and **technical patterns**.

Generate a comprehensive library of **high-frequency alpha factors** covering:
- **Time-based patterns** (opening/closing sessions)
- **Technical indicators** (skewness, volatility, correlations)  
- **Volume microstructure** (clustering, amount-weighted returns)
- **Advanced statistics** (entropy, weighted moments, difference patterns)


- **Input**: Minute-level OHLCV data from datamin2 + datamin3
- **Output**: Daily factor matrices aligned with return data
- **Target**: `../data/factors/obtained_features/`



In [ ]:
# === 1. Import Libraries and Load High-Frequency Data ===

import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import time
import glob
import warnings
warnings.filterwarnings('ignore')

print("📚 Libraries imported successfully")
print("🔄 Starting high-frequency data loading process...")

# === Load Data from Multiple Sources ===
def load_data_from_path(path, description):
    """Load and concatenate all CSV files from a given path"""
    print(f"📊 Loading {description} from: {path}")
    
    try:
        all_files = glob.glob(path + "/*.csv")
        print(f"   Found {len(all_files)} files")
        
        if len(all_files) == 0:
            print(f"   ⚠️  No files found in {path}")
            return None
            
        li = []
        for filename in tqdm(all_files, desc=f"Loading {description}"):
            df = pd.read_csv(filename, index_col=None, header=0)
            li.append(df)
        
        frame = pd.concat(li, axis=0, ignore_index=True)
        result_df = frame.sort_values(by=['order_book_id', 'datetime'], ascending=True)
        result_df = result_df.reset_index(drop=True)
        
        print(f"   ✅ Loaded {len(result_df)} records")
        print(f"   📅 Date range: {result_df['datetime'].min()} to {result_df['datetime'].max()}")
        print(f"   🏢 Unique stocks: {result_df['order_book_id'].nunique()}")
        
        return result_df
        
    except Exception as e:
        print(f"   ❌ Error loading from {path}: {e}")
        return None

# Load data from both sources
path1 = "../data/raw/datamin2"  # Updated path for better organization
path2 = "../data/raw/datamin3"  # Updated path for better organization

result_df1 = load_data_from_path(path1, "datamin2 high-frequency data")
result_df2 = load_data_from_path(path2, "datamin3 high-frequency data")

print(f"\n🔗 Combining datasets...")

In [ ]:
# === 2. Combine Datasets and Prepare for Factor Generation ===

if result_df1 is not None and result_df2 is not None:
    # Combine both datasets
    result_df = pd.concat([result_df1, result_df2], ignore_index=True)
    
    print(f"✅ Datasets combined successfully!")
    print(f"📊 Combined dataset statistics:")
    print(f"   - Total records: {len(result_df):,}")
    print(f"   - Unique stocks: {result_df['order_book_id'].nunique()}")
    print(f"   - Date range: {result_df['datetime'].min()} to {result_df['datetime'].max()}")
    print(f"   - Columns: {list(result_df.columns)}")
    
    # Data quality check
    print(f"\n📈 Data quality assessment:")
    print(f"   - Missing values per column:")
    missing_summary = result_df.isnull().sum()
    for col, missing in missing_summary.items():
        if missing > 0:
            print(f"     {col}: {missing:,} ({missing/len(result_df)*100:.2f}%)")
    
    # Display sample data
    print(f"\n📋 Sample high-frequency data (first 5 rows):")
    print(result_df.head())
    
elif result_df1 is not None:
    print("⚠️  Using only datamin2 data (datamin3 not available)")
    result_df = result_df1
elif result_df2 is not None:
    print("⚠️  Using only datamin3 data (datamin2 not available)")
    result_df = result_df2
else:
    print("❌ No data available - cannot proceed with factor generation")
    result_df = None

,order_book_id,datetime,open,close,high,low,volume,date
0,000001.XSHE,2022-07-05 09:35:00,2219.3295,2248.9403,2260.7846,2219.3295,149188.7101,2022-07-05 00:00:00
1,000001.XSHE,2022-07-05 09:40:00,2247.4597,2248.9403,2253.3819,2240.0570,55486.9020,2022-07-05 00:00:00
2,000001.XSHE,2022-07-05 09:45:00,2250.4208,2248.9403,2251.9013,2244.4986,43954.7001,2022-07-05 00:00:00
3,000001.XSHE,2022-07-05 09:50:00,2247.4597,2247.4597,2251.9013,2245.9792,31647.1671,2022-07-05 00:00:00
4,000001.XSHE,2022-07-05 09:55:00,2247.4597,2241.5376,2247.4597,2238.5765,36178.9038,2022-07-05 00:00:00
...,...,...,...,...,...,...,...,...
81753355,689009.XSHG,2023-12-29 14:40:00,29.7900,29.6800,29.7900,29.6800,135226.0000,2023-12-29 00:00:00
81753356,689009.XSHG,2023-12-29 14:45:00,29.7400,29.8000,29.8700,29.7300,77017.0000,2023-12-29 00:00:00
81753357,689009.XSHG,2023-12-29 14:50:00,29.8000,29.7900,29.8200,29.7600,68245.0000,2023-12-29 00:00:00
81753358,689009.XSHG,2023-12-29 14:55:00,29.7800,29.7000,29.8100,29.6600,137814.0000,2023-12-29 00:00:00


In [ ]:
# === 3. Setup Data Alignment and Export Configuration ===

print("🔧 Setting up data alignment and export configuration...")

# Load return data for date alignment
ret_path = '../data/processed/wide_data_preparation/vwap1pct_daily_data.csv'

try:
    ret_wide = pd.read_csv(ret_path, index_col=0).fillna(0)
    ret_wide.index = pd.to_datetime(ret_wide.index)
    
    print(f"✅ Return data loaded for alignment:")
    print(f"   - Shape: {ret_wide.shape}")
    print(f"   - Date range: {ret_wide.index[0]} to {ret_wide.index[-1]}")
    
    # Create date-to-index mapping for consistent factor export
    date_to_index = {}
    for i in range(len(ret_wide.index)):
        date_to_index[ret_wide.index[i]] = i
    
    print(f"📅 Date mapping created: {len(date_to_index)} trading days")
    
    # Create output directory
    output_dir = "../data/factors/obtained_features"
    os.makedirs(output_dir, exist_ok=True)
    print(f"📁 Output directory ready: {output_dir}")
    
except FileNotFoundError:
    print(f"❌ Error: Return data not found at {ret_path}")
    print("Factor alignment may not work correctly")
    date_to_index = {}
    ret_wide = None

# === Factor Export Helper Function ===
def export_factor(factor_wide, factor_name, description):
    """Helper function to export factors in consistent format"""
    try:
        # Ensure datetime index
        if not isinstance(factor_wide.index, pd.DatetimeIndex):
            factor_wide.index = pd.to_datetime(factor_wide.index)
        
        # Filter to common dates and map to indices
        aligned_factor = factor_wide[factor_wide.index.isin(date_to_index.keys())]
        aligned_factor.index = aligned_factor.index.map(date_to_index)
        aligned_factor = aligned_factor.sort_index()
        
        # Export
        output_path = f"{output_dir}/{factor_name}"
        aligned_factor.to_csv(output_path)
        
        print(f"✅ {description}")
        print(f"   📁 File: {factor_name}")
        print(f"   📊 Shape: {aligned_factor.shape}")
        print(f"   📈 Non-null ratio: {aligned_factor.count().sum()/(aligned_factor.shape[0]*aligned_factor.shape[1]):.2%}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error exporting {factor_name}: {e}")
        return False

print(f"\n🎯 Ready for systematic factor generation...")

In [ ]:
# === 4. Technical Factor 1: Realized Skewness ===

print("📊 Generating Technical Factor 1: Realized Skewness")

if result_df is not None:
    def custom_skew(x):
        """Calculate custom skewness measure for intraday returns"""
        n = x['close'].count()
        if n < 2:
            return np.nan
        numerator = (x['close'] ** 3).sum() * np.sqrt(n)
        denominator = ((x['close'] ** 2).sum()) ** 1.5
        return numerator / denominator if denominator != 0 else np.nan

    print("🔄 Computing realized skewness for each stock-day...")
    
    # Calculate skewness by stock and date
    skewed = result_df.groupby(['order_book_id', 'date']).apply(custom_skew)
    df = skewed.reset_index()
    df.columns = ['order_book_id', 'date', 'skewed'] 
    df = df[['date', 'order_book_id', 'skewed']]
    
    print(f"   ✅ Calculated skewness for {len(df)} stock-day combinations")
    
    # Convert to wide format
    df['date'] = pd.to_datetime(df['date'])
    factor_wide = df.pivot(index="date", columns="order_book_id", values="skewed")
    
    # Export factor
    success = export_factor(factor_wide, "realized_skewness_daily_data.csv", 
                          "Realized Skewness factor exported")
    
    if success:
        print(f"📈 Factor interpretation:")
        print(f"   - Positive skewness: Right-tailed intraday return distribution")
        print(f"   - Negative skewness: Left-tailed intraday return distribution")
        print(f"   - Usage: Capture asymmetric risk patterns in intraday trading")
        
        # Display sample statistics
        print(f"\n📊 Factor statistics:")
        print(f"   - Mean: {factor_wide.stack().mean():.6f}")
        print(f"   - Std: {factor_wide.stack().std():.6f}")
        print(f"   - Min: {factor_wide.stack().min():.6f}")
        print(f"   - Max: {factor_wide.stack().max():.6f}")
    
else:
    print("❌ Cannot generate realized skewness: No data available")

In [ ]:
# === 5. Time-Based Factor 1: Closing Session Volume Ratio ===

print("🕐 Generating Time-Based Factor 1: Closing Session Volume Ratio")

if result_df is not None:
    from datetime import time
    
    print("🔄 Computing closing session (14:00-15:30) volume concentration...")
    
    # Ensure datetime column is properly formatted
    result_df_temp = result_df.copy()
    result_df_temp['datetime'] = pd.to_datetime(result_df_temp['datetime'])
    result_df_temp['time_only'] = result_df_temp['datetime'].dt.time
    
    # Define closing session time window
    start_time = time(14, 0, 0)   # 2:00 PM
    end_time = time(15, 30, 0)    # 3:30 PM
    
    # Filter data for closing session
    filtered = result_df_temp[(result_df_temp['time_only'] >= start_time) & 
                             (result_df_temp['time_only'] <= end_time)]
    
    print(f"   📊 Closing session data: {len(filtered):,} records")
    
    # Calculate volume ratios
    numerator = filtered.groupby(['order_book_id', 'date'])['volume'].sum()
    denominator = result_df_temp.groupby(['order_book_id', 'date'])['volume'].sum()
    at_close_ratio = numerator / denominator
    
    print(f"   ✅ Calculated closing ratios for {len(at_close_ratio)} stock-day combinations")
    
    # Convert to DataFrame and wide format
    at_close_ratio.name = 'at_close_ratio'
    df = at_close_ratio.reset_index()
    df.columns = ['order_book_id', 'date', 'at_close_ratio'] 
    df = df[['date', 'order_book_id', 'at_close_ratio']]
    df['date'] = pd.to_datetime(df['date'])
    
    factor_wide = df.pivot(index="date", columns="order_book_id", values="at_close_ratio")
    
    # Export factor
    success = export_factor(factor_wide, "at_close_ratio_daily_data.csv", 
                          "Closing Session Volume Ratio factor exported")
    
    if success:
        print(f"📈 Factor interpretation:")
        print(f"   - High ratio: Heavy trading during closing session (institutional activity)")
        print(f"   - Low ratio: Even distribution throughout the day")
        print(f"   - Usage: Capture end-of-day trading patterns and institutional behavior")
        
        # Display sample statistics
        print(f"\n📊 Factor statistics:")
        print(f"   - Mean: {factor_wide.stack().mean():.4f}")
        print(f"   - Std: {factor_wide.stack().std():.4f}")
        print(f"   - Range: [{factor_wide.stack().min():.4f}, {factor_wide.stack().max():.4f}]")
    
else:
    print("❌ Cannot generate closing session ratio: No data available")

In [ ]:
# === 6. Comprehensive Alpha Factor Generation ===

print("🚀 Starting comprehensive alpha factor generation process...")

if result_df is not None:
    
    # === Time-Based Factor 2: Opening Session Volume Ratio ===
    print("\n🕐 Generating Opening Session Volume Ratio...")
    
    result_df_temp = result_df.copy()
    result_df_temp['datetime'] = pd.to_datetime(result_df_temp['datetime'])
    result_df_temp['time_only'] = result_df_temp['datetime'].dt.time
    
    start_time = time(10, 0, 0)   # 10:00 AM
    end_time = time(11, 30, 0)    # 11:30 AM
    
    filtered = result_df_temp[(result_df_temp['time_only'] >= start_time) & 
                             (result_df_temp['time_only'] <= end_time)]
    
    numerator = filtered.groupby(['order_book_id', 'date'])['volume'].sum()
    denominator = result_df_temp.groupby(['order_book_id', 'date'])['volume'].sum()
    at_open_ratio = numerator / denominator
    
    df = at_open_ratio.reset_index()
    df.columns = ['order_book_id', 'date', 'at_open_ratio']
    df['date'] = pd.to_datetime(df['date'])
    factor_wide = df.pivot(index="date", columns="order_book_id", values="at_open_ratio")
    
    export_factor(factor_wide, "at_open_ratio_daily_data.csv", 
                  "Opening Session Volume Ratio factor exported")
    
    # === Technical Factor 2: Price-Volume Correlation ===
    print("\n📈 Generating Price-Volume Correlation...")
    
    def fast_corr(close, volume):
        """Fast correlation calculation between price and volume"""
        valid = ~np.isnan(close) & ~np.isnan(volume)
        close = close[valid]
        volume = volume[valid]
        
        if len(close) < 2 or volume.sum() == 0:
            return np.nan

        volume = volume / volume.sum()
        
        close_mean = close.mean()
        volume_mean = volume.mean()

        numerator = np.sum((close - close_mean) * (volume - volume_mean))
        denominator = np.sqrt(np.sum((close - close_mean)**2) * np.sum((volume - volume_mean)**2))
        
        return numerator / denominator if denominator != 0 else np.nan

    grouped = result_df.groupby(['order_book_id', 'date'])
    records = []
    for (order_book_id, date), group in tqdm(grouped, desc="Computing price-volume correlations"):
        corr = fast_corr(group['close'].values, group['volume'].values)
        records.append((order_book_id, date, corr))

    daily_corr_df = pd.DataFrame(records, columns=['order_book_id', 'date', 'close_volume_corr'])
    daily_corr_df['date'] = pd.to_datetime(daily_corr_df['date'])
    factor_wide = daily_corr_df.pivot(index="date", columns="order_book_id", values="close_volume_corr")
    
    export_factor(factor_wide, "high_close_volume_corr_daily_data.csv", 
                  "Price-Volume Correlation factor exported")
    
    # === Volume Factor 1: Top 30% Amount-Weighted Returns ===
    print("\n💰 Generating Top 30% Amount-Weighted Returns...")
    
    def get_top30_factor_wide_fast(df):
        """Calculate top 30% amount-weighted return product"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['amount'] = df['close'] * df['volume']
        df['return_rate'] = (df['close'] - df['open']) / df['open']
        
        df.sort_values(['order_book_id', 'date', 'amount'], ascending=[True, True, False], inplace=True)
        df['rank'] = df.groupby(['order_book_id', 'date']).cumcount()
        df['group_size'] = df.groupby(['order_book_id', 'date'])['amount'].transform('count')
        df['threshold'] = np.ceil(df['group_size'] * 0.3)
        
        top30_df = df[df['rank'] < df['threshold']]
        top30_df['product_component'] = 1 + top30_df['return_rate']
        grouped = top30_df.groupby(['order_book_id', 'date'])['product_component'].prod()
        
        factor_wide = grouped.unstack(level=0)
        return factor_wide

    factor_wide = get_top30_factor_wide_fast(result_df)
    export_factor(factor_wide, "top_30_prod_daily_data.csv", 
                  "Top 30% Amount-Weighted Returns factor exported")
    
    # === Volume Factor 2: Large Volume Clustering ===
    print("\n📊 Generating Large Volume Clustering...")
    
    def get_hug_amount_factor_wide(df):
        """Calculate concentration of large volume trades"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        
        def get_hug_amount(group):
            bar = group['volume'].mean()
            std = group['volume'].std()
            group1 = group[group['volume'] >= bar + std]
            return group1['volume'].sum() / group['volume'].sum()
        
        factor_series = df.groupby(['order_book_id', 'date']).apply(get_hug_amount)
        factor_wide = factor_series.reset_index().pivot(
            index='date', columns='order_book_id', values=0)
        return factor_wide

    factor_wide = get_hug_amount_factor_wide(result_df)
    export_factor(factor_wide, "hug_amount_percent_daily_data.csv", 
                  "Large Volume Clustering factor exported")
    
    # === Technical Factor 3: Intraday Volatility ===
    print("\n📈 Generating Intraday Volatility...")
    
    def get_intraday_volatility_factor_wide(df):
        """Calculate intraday return volatility"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['return_rate'] = (df['close'] - df['open']) / df['open']
        
        factor_series = df.groupby(['order_book_id', 'date'])['return_rate'].std()
        factor_wide = factor_series.reset_index().pivot(
            index='date', columns='order_book_id', values='return_rate')
        return factor_wide

    factor_wide = get_intraday_volatility_factor_wide(result_df)
    export_factor(factor_wide, "intraday_volatility_daily_data.csv", 
                  "Intraday Volatility factor exported")
    
    print(f"\n✅ Comprehensive factor generation completed!")
    print(f"📊 Generated 6 major alpha factors covering:")
    print(f"   - Time-based patterns (opening/closing sessions)")
    print(f"   - Technical indicators (skewness, volatility, correlations)")
    print(f"   - Volume microstructure (clustering, amount-weighted returns)")

else:
    print("❌ Cannot generate factors: No data available")

In [ ]:
# === 7. Advanced Statistical Factors Generation ===

print("🧮 Generating Advanced Statistical Factors...")

if result_df is not None:
    
    # === Advanced Factor 1: Top 20% Amount-Weighted Returns ===
    print("\n💰 Generating Top 20% Amount-Weighted Returns...")
    
    def get_top20_factor_wide_fast(df):
        """Calculate top 20% amount-weighted return product"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['amount'] = df['close'] * df['volume']
        df['return_rate'] = (df['close'] - df['open']) / df['open']
        
        df.sort_values(['order_book_id', 'date', 'amount'], ascending=[True, True, False], inplace=True)
        df['rank'] = df.groupby(['order_book_id', 'date']).cumcount()
        df['group_size'] = df.groupby(['order_book_id', 'date'])['amount'].transform('count')
        df['threshold'] = np.ceil(df['group_size'] * 0.2)
        
        top20_df = df[df['rank'] < df['threshold']]
        top20_df['product_component'] = 1 + top20_df['return_rate']
        grouped = top20_df.groupby(['order_book_id', 'date'])['product_component'].prod()
        
        factor_wide = grouped.unstack(level=0)
        return factor_wide

    factor_wide = get_top20_factor_wide_fast(result_df)
    export_factor(factor_wide, "top_20_prod_daily_data.csv", 
                  "Top 20% Amount-Weighted Returns factor exported")
    
    # === Advanced Factor 2: Volume-Weighted Close Factor ===
    print("\n📊 Generating Volume-Weighted Close Factor...")
    
    def get_weighted_close(df):
        """Calculate volume-weighted close factor"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['weighted_value'] = df['close'] * df['volume']
        df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('mean')
        df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
        df['weighted_close'] = df['weighted_value'] / (df['vol'] * df['d2'])
        
        factor_wide = df.pivot_table(index='date', columns='order_book_id', values='weighted_close')
        return factor_wide

    factor_wide = get_weighted_close(result_df)
    export_factor(factor_wide, "weighted_close_daily_data.csv", 
                  "Volume-Weighted Close factor exported")
    
    # === Advanced Factor 3: Information Entropy ===
    print("\n🔬 Generating Information Entropy Factor...")
    
    def get_entropy(df):
        """Calculate information entropy based on volume-price distribution"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('sum')
        df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
        df['single'] = (df['volume'] / df['vol']) * (df['close'] / df['d2'])
        df['d3'] = -df['single'] * np.log(df['single'] + 1e-10)  # Add small epsilon to avoid log(0)
        df['d1'] = df.groupby(['order_book_id', 'date'])['d3'].transform('sum')
        
        factor_wide = df.pivot_table(index='date', columns='order_book_id', values='d1')
        return factor_wide

    factor_wide = get_entropy(result_df)
    export_factor(factor_wide, "entropy_daily_weighted_close_data.csv", 
                  "Information Entropy factor exported")
    
    # === Advanced Factor 4: Volume Difference Statistics ===
    print("\n📈 Generating Volume Difference Statistics...")
    
    def get_diff_std(df):
        """Calculate volume difference standard deviation"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['diff'] = df.groupby(['order_book_id', 'date'])['volume'].diff()
        df['nom1'] = df.groupby(['order_book_id', 'date'])['diff'].transform('std')
        df['nom2'] = df.groupby(['order_book_id', 'date'])['volume'].transform('mean')
        df['d1'] = df['nom1'] / df['nom2']
        
        factor_wide = df.pivot_table(index='date', columns='order_book_id', values='d1')
        return factor_wide

    factor_wide = get_diff_std(result_df)
    export_factor(factor_wide, "diff_std_data.csv", 
                  "Volume Difference Std factor exported")
    
    def get_diff_mean(df):
        """Calculate volume difference mean"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['diff'] = df.groupby(['order_book_id', 'date'])['volume'].diff()
        df['nom2'] = df.groupby(['order_book_id', 'date'])['volume'].transform('mean')
        df['d3'] = np.abs(df['diff'] / df['nom2'])
        df['d1'] = df.groupby(['order_book_id', 'date'])['d3'].transform('mean')
        
        factor_wide = df.pivot_table(index='date', columns='order_book_id', values='d1')
        return factor_wide

    factor_wide = get_diff_mean(result_df)
    export_factor(factor_wide, "diff_mean_data.csv", 
                  "Volume Difference Mean factor exported")
    
    # === Advanced Factor 5: Amount-based Difference Statistics ===
    print("\n💱 Generating Amount-based Difference Statistics...")
    
    def get_amount_diff_std(df):
        """Calculate amount difference standard deviation"""
        df = df.copy()
        df['amount'] = df['volume'] * df['close']
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['diff'] = df.groupby(['order_book_id', 'date'])['amount'].diff()
        df['nom1'] = df.groupby(['order_book_id', 'date'])['diff'].transform('std')
        df['nom2'] = df.groupby(['order_book_id', 'date'])['amount'].transform('mean')
        df['d1'] = df['nom1'] / df['nom2']
        
        factor_wide = df.pivot_table(index='date', columns='order_book_id', values='d1')
        return factor_wide

    factor_wide = get_amount_diff_std(result_df)
    export_factor(factor_wide, "diff_std_amount_data.csv", 
                  "Amount Difference Std factor exported")
    
    print(f"\n✅ Advanced statistical factors generation completed!")
    print(f"📊 Generated 7 additional sophisticated factors")

else:
    print("❌ Cannot generate advanced factors: No data available")

In [ ]:
# === 8. Final Summary and Factor Library Report ===

print("="*80)
print("📋 ALPHA FACTOR GENERATION 2 - SUMMARY REPORT")
print("="*80)

print(f"\n🎯 METHODOLOGY:")
print(f"   Data Source: High-frequency intraday minute-level OHLCV data")
print(f"   Processing: Combined datamin2 and datamin3 datasets")
print(f"   Approach: Technical & microstructure-based factor engineering")
print(f"   Output Format: Wide format (dates × stocks) aligned with return data")

print(f"\n📊 GENERATED FACTOR LIBRARY:")

factor_categories = {
    "🕐 Time-Based Factors": [
        "at_close_ratio_daily_data.csv - Closing session volume concentration",
        "at_open_ratio_daily_data.csv - Opening session volume concentration"
    ],
    "📈 Technical & Statistical Factors": [
        "realized_skewness_daily_data.csv - Intraday return distribution asymmetry",
        "high_close_volume_corr_daily_data.csv - Price-volume correlation",
        "intraday_volatility_daily_data.csv - Intraday return volatility"
    ],
    "💰 Volume & Amount Factors": [
        "top_30_prod_daily_data.csv - Top 30% amount-weighted return product",
        "top_20_prod_daily_data.csv - Top 20% amount-weighted return product",
        "hug_amount_percent_daily_data.csv - Large volume trade clustering"
    ],
    "🧮 Advanced Statistical Factors": [
        "weighted_close_daily_data.csv - Volume-weighted close deviation",
        "entropy_daily_weighted_close_data.csv - Information entropy measure",
        "diff_std_data.csv - Volume difference standard deviation",
        "diff_mean_data.csv - Volume difference mean",
        "diff_std_amount_data.csv - Amount difference standard deviation"
    ]
}

total_factors = 0
for category, factors in factor_categories.items():
    print(f"\n{category}:")
    for factor in factors:
        print(f"   • {factor}")
        total_factors += 1

print(f"\n📁 OUTPUT LOCATION:")
print(f"   Directory: ../data/factors/obtained_features/")
print(f"   Total Factors Generated: {total_factors}")
print(f"   Format: CSV files with date indices aligned to return data")

print(f"\n🔬 RESEARCH APPLICATIONS:")
print(f"   • Market Microstructure Analysis: Time-based and volume clustering patterns")
print(f"   • Technical Analysis: Statistical moments and price-volume relationships") 
print(f"   • Risk Management: Volatility and distribution asymmetry measures")
print(f"   • Alpha Generation: Sophisticated quantile-based and entropy factors")

print(f"\n🎯 NEXT STEPS:")
print(f"   1. Run factor backtesting using factor_backtest.ipynb")
print(f"   2. Evaluate IC and performance metrics for each factor")
print(f"   3. Include promising factors in Alpha_Factor_Selection.ipynb")
print(f"   4. Test factor combinations and portfolio construction")
print(f"   5. Optimize parameters for different market regimes")

print(f"\n🚀 HIGH-FREQUENCY ALPHA FACTOR GENERATION COMPLETED!")
print(f"   Ready for systematic evaluation and strategy implementation")

print(f"\n" + "="*80)

,order_book_id,date,close_volume_corr
0,000001.XSHE,2022-07-05 00:00:00,0.201478
1,000001.XSHE,2022-07-06 00:00:00,0.462733
2,000001.XSHE,2022-07-07 00:00:00,-0.238029
3,000001.XSHE,2022-07-08 00:00:00,-0.502432
4,000001.XSHE,2022-07-11 00:00:00,0.328913
...,...,...,...
1703190,689009.XSHG,2023-12-25 00:00:00,-0.079361
1703191,689009.XSHG,2023-12-26 00:00:00,-0.306593
1703192,689009.XSHG,2023-12-27 00:00:00,0.428902
1703193,689009.XSHG,2023-12-28 00:00:00,-0.219729


In [77]:
daily_corr_df['date']=pd.to_datetime(daily_corr_df['date'])
size_wide_daily = daily_corr_df.pivot(index="date",columns="order_book_id",values="close_volume_corr")
size_wide_daily = size_wide_daily[size_wide_daily.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/high_close_volume_corr_daily_data.csv")

In [83]:
def get_top30_factor_wide_fast(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['amount'] = df['close'] * df['volume']
    df['return_rate'] = (df['close'] - df['open']) / df['open']
    df.sort_values(['order_book_id', 'date', 'amount'], ascending=[True, True, False], inplace=True)
    df['rank'] = df.groupby(['order_book_id', 'date']).cumcount()
    df['group_size'] = df.groupby(['order_book_id', 'date'])['amount'].transform('count')
    df['threshold'] = np.ceil(df['group_size'] * 0.3)
    top30_df = df[df['rank'] < df['threshold']]
    top30_df['product_component'] = 1 + top30_df['return_rate']
    grouped = top30_df.groupby(['order_book_id', 'date'])['product_component'].prod()
    factor_wide = grouped.unstack(level=0)
    factor_wide.name = 'top30_product' 
    return factor_wide

In [84]:
df=get_top30_factor_wide_fast(result_df)

order_book_id  000001.XSHE  000002.XSHE  000004.XSHE  000005.XSHE  \
date                                                                
2022-07-05        1.014752     1.006995     0.978691     1.018290   
2022-07-06        0.980432     0.972911     1.059551     0.999846   
2022-07-07        0.990349     0.987588     1.009006     1.000042   
2022-07-08        1.004119     1.010932     0.969517     0.993930   
2022-07-11        0.995170     0.992694     1.000000     1.006095   
...                    ...          ...          ...          ...   
2023-12-25        0.996739     0.996132     0.994343     0.981129   
2023-12-26        0.992387     0.988346     0.985976     1.040193   
2023-12-27        1.001084     0.989226     1.022086     1.000000   
2023-12-28        1.046328     1.039448     0.987818     1.047805   
2023-12-29        0.993634     0.998083     1.029239     1.009256   

order_book_id  000006.XSHE  000007.XSHE  000008.XSHE  000009.XSHE  \
date                             

In [87]:
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/top_30_prod_daily_data.csv")

In [88]:
size_wide_daily

order_book_id,000001.XSHE,000002.XSHE,000004.XSHE,000005.XSHE,000006.XSHE,000007.XSHE,000008.XSHE,000009.XSHE,000010.XSHE,000011.XSHE,...,688787.XSHG,688788.XSHG,688789.XSHG,688793.XSHG,688798.XSHG,688799.XSHG,688800.XSHG,688819.XSHG,688981.XSHG,689009.XSHG
date,,,,,,,,,,,,,,,,,,,,,
0,1.014752,1.006995,0.978691,1.018290,1.006602,0.974297,1.008147,0.980402,0.974219,0.984703,...,0.934371,0.927359,1.010126,0.954498,0.967356,0.973877,0.977891,0.971155,0.998602,0.967187
1,0.980432,0.972911,1.059551,0.999846,0.991121,1.006709,0.975530,0.957407,0.965833,0.974662,...,1.042370,1.014538,1.027329,0.996731,1.033071,0.978467,0.974712,1.010638,0.996796,0.977654
2,0.990349,0.987588,1.009006,1.000042,1.009065,0.992181,1.012483,1.008781,0.991727,1.013351,...,1.005349,0.984920,0.986670,0.953029,1.003057,0.991712,1.118974,0.997049,0.999733,0.979394
3,1.004119,1.010932,0.969517,0.993930,1.002171,0.997392,0.991757,0.980676,1.018742,1.015636,...,0.955999,1.059739,1.080848,0.984540,0.992843,1.001998,0.928651,1.000143,1.003903,1.033032
4,0.995170,0.992694,1.000000,1.006095,0.995402,0.987237,1.012394,0.971483,0.957717,0.966223,...,0.986888,0.974823,1.007386,1.003767,0.987991,1.013911,1.105916,0.975813,0.970935,1.069538
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319,1.000003,0.986923,1.004441,0.993161,0.970170,0.990008,1.004050,1.013005,0.970109,0.989997,...,0.958062,0.986701,0.983297,1.011279,0.992732,1.008853,0.972305,0.993600,1.023210,1.022893
320,0.990093,0.966888,0.998881,0.979021,0.989424,0.995959,1.000001,1.000883,0.997095,0.975473,...,1.017441,1.012599,1.050020,0.980603,0.999596,0.990471,1.022404,0.983772,1.012768,1.005909
321,1.002999,1.006797,0.990543,0.985800,1.012867,0.981899,1.016310,1.010289,1.000020,0.988497,...,0.994083,1.012686,0.990631,1.009338,0.989972,1.013015,1.010792,0.992733,1.001643,0.975366


In [89]:
def get_hug_amount_factor_wide(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    def get_hug_amount(group):
        bar=group['volume'].mean()
        std=group['volume'].std()
        group1=group[group['volume']>=bar+std]
        return group1['volume'].sum()/group['volume'].sum()
    factor_series = df.groupby(['order_book_id', 'date']).apply(get_hug_amount)
    factor_series.name = 'hug_amount'
    factor_wide = factor_series.reset_index().pivot(
        index='date', 
        columns='order_book_id', 
        values='hug_amount'
    )
    return factor_wide
factor_wide = get_hug_amount_factor_wide(result_df)

In [91]:
factor_wide.index = pd.to_datetime(factor_wide.index)
size_wide_daily = factor_wide[factor_wide.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/hug_amount_percent_daily_data.csv")

In [92]:
def get_intraday_volatility_factor_wide(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['return_rate'] = (df['close'] - df['open']) / df['open']
    factor_series = df.groupby(['order_book_id', 'date'])['return_rate'].std()
    factor_series.name = 'intraday_volatility'
    factor_wide = factor_series.reset_index().pivot(
        index='date',
        columns='order_book_id',
        values='intraday_volatility'
    )
    return factor_wide
factor_wide = get_intraday_volatility_factor_wide(result_df)

In [93]:
factor_wide.index = pd.to_datetime(factor_wide.index)
size_wide_daily = factor_wide[factor_wide.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/intraday_volatility_daily_data.csv")

In [94]:
def get_top20_factor_wide_fast(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['amount'] = df['close'] * df['volume']
    df['return_rate'] = (df['close'] - df['open']) / df['open']
    df.sort_values(['order_book_id', 'date', 'amount'], ascending=[True, True, False], inplace=True)
    df['rank'] = df.groupby(['order_book_id', 'date']).cumcount()
    df['group_size'] = df.groupby(['order_book_id', 'date'])['amount'].transform('count')
    df['threshold'] = np.ceil(df['group_size'] * 0.2)
    top20_df = df[df['rank'] < df['threshold']]
    top20_df['product_component'] = 1 + top20_df['return_rate']
    grouped = top20_df.groupby(['order_book_id', 'date'])['product_component'].prod()
    factor_wide = grouped.unstack(level=0)
    factor_wide.name = 'top20_product' 
    return factor_wide

In [95]:
df=get_top20_factor_wide_fast(result_df)
df

order_book_id,000001.XSHE,000002.XSHE,000004.XSHE,000005.XSHE,000006.XSHE,000007.XSHE,000008.XSHE,000009.XSHE,000010.XSHE,000011.XSHE,...,688787.XSHG,688788.XSHG,688789.XSHG,688793.XSHG,688798.XSHG,688799.XSHG,688800.XSHG,688819.XSHG,688981.XSHG,689009.XSHG
date,,,,,,,,,,,,,,,,,,,,,
2022-07-05,1.012723,1.008965,0.961340,1.030414,1.002183,0.979335,0.999967,0.981152,0.974246,0.977269,...,0.932852,0.924743,1.004482,0.953753,0.971836,0.964897,0.985062,0.986154,0.999048,0.974340
2022-07-06,0.981690,0.968984,1.052916,0.993788,0.982301,1.001306,0.987689,0.964605,0.986553,0.977674,...,1.045686,1.016432,1.012489,0.988454,1.025692,0.976605,0.956841,1.012381,1.000650,0.978561
2022-07-07,0.990346,0.990194,1.045627,0.994092,1.004519,0.994748,1.012483,1.007268,0.989028,1.014136,...,1.017839,0.986578,0.977763,0.941063,1.000779,0.984863,1.071487,0.976273,0.999282,0.976002
2022-07-08,0.999281,1.017784,0.969517,0.999958,1.006666,0.987058,1.012179,0.990178,1.024133,1.019602,...,0.954149,1.043234,1.053558,0.999976,1.006652,1.000668,0.953603,0.999320,1.005519,1.042896
2022-07-11,0.994476,0.988042,1.000000,1.000000,0.988698,0.988533,1.020725,0.967642,0.957743,0.961277,...,0.972755,0.975260,1.021675,1.018883,0.990323,1.010588,1.062168,0.988229,0.969792,1.058270
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-25,0.994562,0.998063,0.985636,0.971779,0.993254,0.981400,0.974612,1.008625,0.993561,0.974141,...,1.021577,1.029378,0.990113,1.012253,1.002710,0.929635,1.026267,1.001079,0.980462,0.993219
2023-12-26,0.996743,0.988343,0.991059,1.020389,0.997769,0.979245,0.995690,0.999997,1.000043,1.001133,...,0.994950,0.965840,0.998947,0.989887,0.989922,0.971388,0.979147,1.003275,0.979457,0.949343
2023-12-27,0.997795,0.997047,1.024056,1.029411,1.008859,0.989535,1.004309,1.004306,1.019818,1.001145,...,1.012408,1.018254,0.999702,0.991985,1.011948,1.010845,0.967584,0.985408,0.997479,1.027143


In [96]:
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/top_20_prod_daily_data.csv")

In [115]:
def get_weighted_close(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['weighted_value'] = df['close'] * df['volume']
    df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('mean')
    df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
    df['weighted_close'] = df['weighted_value'] / (df['vol'] * df['d2'])
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='weighted_close'
    )
    
    return factor_wide

In [116]:
df=get_weighted_close(result_df)

In [118]:
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/weighted_close_daily_data.csv")

In [119]:
def get_weighted_sig(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('mean')
    df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
    df['std3'] = df.groupby(['order_book_id', 'date'])['close'].transform('std')**3
    df['d1'] = (df['volume'] / df['vol']) * ((df['close'] - df['d2'])**3) / df['std3']
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    return factor_wide

In [120]:
df=get_weighted_sig(result_df)

In [123]:
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/weighted_close_daily_sig_data.csv")

In [124]:
def get_weighted_sig1(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('mean')
    df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
    df['std3'] = df.groupby(['order_book_id', 'date'])['close'].transform('std')**4
    df['d1'] = (df['volume'] / df['vol']) * ((df['close'] - df['d2'])**4) / df['std3']
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    return factor_wide
df=get_weighted_sig1(result_df)
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/weighted_close_daily_sig1_data.csv")

In [125]:
def get_weighted_sig2(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('mean')
    df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
    df['std3'] = df.groupby(['order_book_id', 'date'])['close'].transform('std')**5
    df['d1'] = (df['volume'] / df['vol']) * ((df['close'] - df['d2'])**5) / df['std3']
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    return factor_wide
df=get_weighted_sig2(result_df)
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/weighted_close_daily_sig2_data.csv")

In [126]:
def get_weighted_sig3(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('mean')
    df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
    df['std3'] = df.groupby(['order_book_id', 'date'])['close'].transform('std')**6
    df['d1'] = (df['volume'] / df['vol']) * ((df['close'] - df['d2'])**6) / df['std3']
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    return factor_wide
df=get_weighted_sig3(result_df)
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/weighted_close_daily_sig3_data.csv")

In [127]:
def get_entropy(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('sum')
    df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
    df['single']=(df['volume']/df['vol'])*(df['close']/df['d2'])
    df['d3'] = -df['single']*np.log(df['single'])
    df['d1']=df.groupby(['order_book_id','date'])['d3'].transform('sum')
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    return factor_wide
df=get_entropy(result_df)
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/entropy_daily_weighted_close_data.csv")

In [128]:
def get_weighted_sig4(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('mean')
    df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
    df['std3'] = df.groupby(['order_book_id', 'date'])['close'].transform('std')**7
    df['d1'] = (df['volume'] / df['vol']) * ((df['close'] - df['d2'])**7) / df['std3']
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    return factor_wide
df=get_weighted_sig4(result_df)
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/weighted_close_daily_sig4_data.csv")

In [129]:
def get_weighted_sig5(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('mean')
    df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
    df['std3'] = df.groupby(['order_book_id', 'date'])['close'].transform('std')**8
    df['d1'] = (df['volume'] / df['vol']) * ((df['close'] - df['d2'])**8) / df['std3']
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    return factor_wide
df=get_weighted_sig5(result_df)
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/weighted_close_daily_sig5_data.csv")

In [132]:
def get_diff_std(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['diff'] = df.groupby(['order_book_id', 'date'])['volume'].diff()
    df['nom1'] = df.groupby(['order_book_id', 'date'])['diff'].transform('std')
    df['nom2'] = df.groupby(['order_book_id', 'date'])['volume'].transform('mean')
    df['d1'] = df['nom1'] / df['nom2']
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    
    return factor_wide
df=get_diff_std(result_df)
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/diff_std_data.csv")

In [9]:
def get_diff_mean(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['diff'] = df.groupby(['order_book_id', 'date'])['volume'].diff()
    df['nom2'] = df.groupby(['order_book_id', 'date'])['volume'].transform('mean')
    df['d3'] = np.abs(df['diff'] / df['nom2'])
    df['d1'] = df.groupby(['order_book_id', 'date'])['d3'].transform('mean')
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    return factor_wide
df=get_diff_mean(result_df)
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/diff_mean_data.csv")

In [6]:
def get_amount_diff_std(df):
    df = df.copy()
    df['amount']=df['volume']*df['close']
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['diff'] = df.groupby(['order_book_id', 'date'])['amount'].diff()
    df['nom1'] = df.groupby(['order_book_id', 'date'])['diff'].transform('std')
    df['nom2'] = df.groupby(['order_book_id', 'date'])['amount'].transform('mean')
    df['d1'] = df['nom1'] / df['nom2']
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    
    return factor_wide
df=get_amount_diff_std(result_df)
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/diff_std_amount_data.csv")

In [7]:
def get_amount2_diff_std(df):
    df = df.copy()
    df['amount']=df['volume']*df['close']
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['diff'] = df.groupby(['order_book_id', 'date'])['amount'].diff(2)
    df['nom1'] = df.groupby(['order_book_id', 'date'])['diff'].transform('std')
    df['nom2'] = df.groupby(['order_book_id', 'date'])['amount'].transform('mean')
    df['d1'] = df['nom1'] / df['nom2']
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    
    return factor_wide
df=get_amount2_diff_std(result_df)
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/diff_std_amount2_data.csv")

In [10]:
def get_diff2_mean(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.date
    df['diff'] = df.groupby(['order_book_id', 'date'])['volume'].diff(2)
    df['nom2'] = df.groupby(['order_book_id', 'date'])['volume'].transform('mean')
    df['d3'] = np.abs(df['diff'] / df['nom2'])
    df['d1'] = df.groupby(['order_book_id', 'date'])['d3'].transform('mean')
    factor_wide = df.pivot_table(
        index='date', 
        columns='order_book_id', 
        values='d1'
    )
    return factor_wide
df=get_diff2_mean(result_df)
df.index = pd.to_datetime(df.index)
size_wide_daily = df[df.index.isin(dict)]
size_wide_daily.index = size_wide_daily.index.map(dict)
size_wide_daily.to_csv("obtained_features/diff2_mean_data.csv")